# FloodScout — train the flood model

Trains the model FloodScout scores regions with, on Google Colab.

**What it learns from.** One row per Kenyan county per month, 1985–2021: rainfall for the 30 days ending that month (ERA5 reanalysis), the El Niño index (NOAA ONI), elevation and slope, and whether the Dartmouth Flood Observatory recorded a large flood there in that window.

**What it outputs.** The probability that a large flood is recorded in a region in a 30-day window.

The training code is not in this notebook. It is `riskengine/ml/train.py` from the repository, uploaded in the bundle below, so Colab and the app always run the same code.

**Before you start**, build the bundle on your machine from the repository root:

```
docker compose run --rm web python manage.py build_training_set
tar -czf data/processed/colab_bundle.tar.gz data/processed/training_set.csv riskengine/__init__.py riskengine/ml/__init__.py riskengine/ml/train.py riskengine/ml/features.py geodata/__init__.py geodata/services/__init__.py geodata/services/climate_indices.py
```

## 1. Match the app's scikit-learn

The saved model is a pickle. It only loads reliably under the scikit-learn version that wrote it, so this must match `requirements.txt`. If Colab asks to restart the session after this cell, restart and carry on from step 2.

In [ ]:
%pip install -q "scikit-learn==1.9.1"
import sklearn
print(sklearn.__version__)

## 2. Upload `colab_bundle.tar.gz`

It is in `data/processed/` in the repository.

In [ ]:
from google.colab import files

uploaded = files.upload()
bundle = next(iter(uploaded))
!tar -xzf "$bundle"
!ls data/processed riskengine/ml geodata/services

## 3. Train

Compares Logistic Regression, Random Forest and Gradient Boosting, holding out whole years at a time, then refits the best one on all the data. Takes a minute or two.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, ".")
from geodata.services.climate_indices import fetch_oni
from riskengine.ml import train

artifact, metrics = train.train(Path("data/processed/training_set.csv"), fetch_oni())
train.save(artifact, metrics)
print("Trained", artifact["version"])

## 4. How good is it?

All figures are on years the model did not see. **Average precision** is the one the winner is picked on; compare it with the base rate, which is what a model that knows nothing would score. **Brier** is lower-is-better; compare it with the baseline row.

In [ ]:
import pandas as pd

print(f"{metrics['rows']} county-months, {metrics['positives']} with a recorded flood "
      f"(base rate {metrics['base_rate']:.1%}); selected: {metrics['selected']}")
comparison = pd.DataFrame(metrics["comparison"]).T
comparison.loc["(always base rate)"] = [0.5, metrics["base_rate"], metrics["baseline_brier"]]
comparison

How often a flood was actually recorded in each risk band, on held-out years. The bands are only useful if this rises from Low to Very High.

In [ ]:
pd.DataFrame(metrics["band_calibration"]).set_index("band")

What pushes the probability up or down (logistic regression only). Coefficients are per standard deviation of each input, so they can be compared with each other.

In [ ]:
pd.Series(metrics.get("standardised_coefficients", {}), name="coefficient").sort_values().to_frame()

## 5. Download the model

Put both files in `riskengine/ml/artifacts/` in the repository, replacing what is there. Then:

```
docker compose restart web
docker compose run --rm web python manage.py seed_rainfall
docker compose run --rm web python manage.py score_regions
docker compose run --rm web python manage.py flood_outlook
```

In [ ]:
files.download(str(train.ARTIFACT_PATH))
files.download(str(train.METRICS_PATH))